# IEEE 33-Bus Optimal Power Flow (OPF)

This notebook solves an optimal power flow for the IEEE 33-bus distribution system. It is the optimization module for the WIRED Grid Data Challenge.

**What is an OPF?** A power flow works out the voltages and line flows in a network for a given set of loads. An *optimal* power flow does the same thing, but also lets an optimizer choose the values of anything that can be controlled so that some cost is as small as possible, while every physical equation and operating limit is still respected.

**What this notebook does:**

- Builds the IEEE 33-bus feeder from its standard benchmark data (line impedances and bus loads).
- Writes the power flow equations using the **branch flow model** (also called DistFlow).
- Minimizes the **total active power losses** in the lines.
- Solves over **24 hourly time steps**. Every equation is indexed by time, so you can change the horizon.
- Saves bus voltages, line flows, substation power and losses, and plots the voltage at every bus over the day.

There is no PV, no storage and no external load data in this version. Every hour uses the rated (nominal) load, so all 24 hours give the same answer. This is the base case you will build on later.

## How to Run This Notebook

This notebook is self-contained: it does not need any other files.

### 1. Install the Python packages
Run in a terminal (or Anaconda Prompt):
```
pip install pyomo numpy pandas scipy matplotlib ipykernel
```

### 2. Install the IPOPT solver
Pyomo does **not** include the IPOPT solver; it must be installed separately.
The solver setup cell after the imports installs IPOPT automatically if it is missing. To install it yourself instead, use **one** of these options:

- **With Anaconda:**
  ```
  conda install -c conda-forge ipopt
  ```
- **Without conda:**
  ```
  pip install idaes-pse
  idaes get-extensions
  ```

### 3. Select the kernel
In Jupyter or VS Code, select the Python environment where you installed the packages and IPOPT in steps 1 and 2.

### 4. Run the cells
Run the cells **in order from top to bottom** (Shift+Enter), or use **Run All**. Each cell depends on the cells above it.
If you change a value in the **Configuration** cell (e.g. `N`, `Vmag`), re-run that cell and all cells below it.

### Outputs
Results are saved to `Results/Obj(MinLosses)/Results_<Vmag>pu/` in the same folder as the notebook:
- `Voltage_Profiles_Data.csv`: bus voltages (p.u.) for each hour
- `Pg_Data.pkl`, `Total_losses_Data.pkl`, `Branches_Data.pkl`: substation power, total losses, and branch flows
- `Voltage_Profiles.pdf`: voltage profile figure


## Imports

The libraries used in this notebook:

| Library | What it is used for |
|---|---|
| `matplotlib` | Plotting the voltage profiles |
| `pandas` | Putting the voltage results in a table and saving it as a CSV |
| `pickle` | Saving Python dictionaries and lists to files so they can be loaded later |
| `os` | Creating the output folder |
| `scipy.interpolate.interp1d` | Smoothing the hourly results into a 1-minute curve for the plot |
| `pyomo.environ` | The optimization modelling language. You write variables, constraints and the objective in Python, and Pyomo passes them to a solver |

Pyomo only describes the problem. The actual solving is done by a separate program called a **solver**. This notebook uses **IPOPT**, which handles nonlinear problems like this one.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import os
from scipy.interpolate import interp1d
from pyomo.environ import *

### Solver Setup

This cell checks whether IPOPT is installed. If it isn't, it installs it, using `conda` if conda is available and the IDAES package otherwise. This only takes time the first time you run it. If the last line prints `IPOPT available: True`, you're ready to go.

In [ ]:
# Solver setup: installs IPOPT if it is not already available (one-time, about 1 minute)
import shutil, subprocess, sys
if not SolverFactory('ipopt').available(exception_flag=False):
    if shutil.which('conda'):
        subprocess.run(['conda', 'install', '-y', '-q', '-c', 'conda-forge', 'ipopt'], check=True)
    else:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'idaes-pse'], check=True)
        subprocess.run(['idaes', 'get-extensions'], check=True)
        import idaes  # puts the IDAES binary folder (with ipopt) on PATH
print('IPOPT available:', SolverFactory('ipopt').available(exception_flag=False))

## Configuration

These are the settings for the whole notebook. If you change anything here, re-run this cell and every cell below it.

**Base values and per unit.** Power system calculations are usually done in *per unit* (p.u.): each quantity is divided by a chosen base value, so most numbers end up close to 1. For example, a voltage of 0.95 p.u. means 95% of the rated voltage. This keeps the numbers in a similar range, which makes the solver more reliable.

- `V_base = 12.66 kV` is the rated line-to-line voltage of the IEEE 33-bus system.
- `S_base = 1 MVA` is the base power.
- `I_base = S_base / (sqrt(3) * V_base)` is the base current, about 45.6 A. The sqrt(3) is there because this is a three-phase system and `V_base` is a line-to-line voltage.

**Voltage settings.**

- `Vmag` is the voltage at the substation (bus 0), fixed at 1.0 p.u. The substation is the *slack bus*: it holds the voltage reference and supplies whatever power the feeder needs.
- `V_min = 0.90` and `V_max = 1.05` are the allowed voltage limits at every bus.

**Time steps.** `N = days * 24` is the number of hourly time steps. With `days = 1` the model solves 24 hours.

In [ ]:
import numpy as np

# Base Values (IEEE 33-bus, Baran & Wu 1989)
V_base = 12.66e3  # line to line
S_base = 1e6
I_base = S_base / (V_base * np.sqrt(3))

# Substation (slack) voltage magnitude, p.u.
Vmag = 1.0

# Bus voltage limits, p.u.
V_min = 0.90
V_max = 1.05

# Time series: number of time steps
days=1
N = days*24

## Network Data

This cell holds all the data that describes the IEEE 33-bus system. The values come from the original benchmark paper by Baran and Wu (1989).

Buses are numbered **0 to 32** in this code. Bus 0 is the substation. Most papers and diagrams number the buses 1 to 33, so bus 17 here is bus 18 in a paper.

**`t`** is the list of time steps (0 to N-1).

**`flow_vars`** is the list of the 32 lines, written as `(from bus, to bus)`. Power flows from the substation outwards, so the "from" bus is always the one closer to the substation. The layout is a radial tree:

```
0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17     main feeder
  | |       |
  | |       5-25-26-27-28-29-30-31-32           lateral from bus 5
  | 2-22-23-24                                  lateral from bus 2
  1-18-19-20-21                                 lateral from bus 1
```

**`Z_Nominal`** is the impedance of each line in ohms, written as `R + jX`. R is the resistance, which causes the losses. X is the reactance, which mostly affects the voltage drop. `Z_base = V_base^2 / S_base` (about 160 ohms) is used to convert them to per unit, and the per-unit values are stored in `Z`.

**`bus_vars`** is the list of buses, 0 to 32.

**`bus2flow`** tells the model which lines are connected to each bus. For example, bus 1 is connected to lines (0,1), (1,2) and (1,18). The power balance equations use this to know what flows into and out of each bus.

**`load_p_nominal` and `load_q_nominal`** are the loads at each bus. P is active power in watts, the power that actually does work. Q is reactive power in var, drawn by things like motors and transformers. It does no useful work, but it still has to flow through the lines. The total load is 3.715 MW and 2.30 Mvar.

**`load_p` and `load_q`** convert the loads to per unit and repeat them for every time step. This is the part you would change to use a real load profile, so that each hour has a different load.

In [ ]:
# This file stores the line and load data for the IEEE 33-bus system 

# time index
t = np.arange(0, N, 1)

# Line flow variables 
flow_vars =[ (0,1),
             (1,2),
             (2,3),
             (3,4),
             (4,5),
             (5,6),
             (6,7),
             (7,8),
             (8,9),
             (9,10),
             (10,11),
             (11,12),
             (12,13),
             (13,14),
             (14,15),
             (15,16),
             (16,17),
             (1,18),
             (18,19),
             (19,20), 
             (20,21),
             (2,22),
             (22,23),
             (23,24), 
             (5,25),
             (25,26),
             (26,27),
             (27,28),
             (28,29),
             (29,30),
             (30,31),
             (31,32)]


# line impedances
Z_base = (V_base ** 2) / S_base


Z_Nominal = {key: val for key, val in {
            (0,1)  : 0.0922+ 0.0470j,
            (1,2)  : 0.4930+ 0.2511j,
            (2,3)  : 0.3660+ 0.1864j,
            (3,4)  : 0.3811+ 0.1941j,
            (4,5)  : 0.8190+ 0.7070j,
            (5,6)  : 0.1872+ 0.6188j,
            (6,7)  : 0.7114+ 0.2351j,
            (7,8)  : 1.0300+ 0.7400j,
            (8,9)  : 1.0440+ 0.7400j,
            (9,10) : 0.1966+ 0.0650j,
            (10,11): 0.3744+ 0.1238j,
            (11,12): 1.4680+ 1.1550j,
            (12,13): 0.5416+ 0.7129j,
            (13,14): 0.5910+ 0.5260j,
            (14,15): 0.7463+ 0.5450j,
            (15,16): 1.2890+ 1.7210j,
            (16,17): 0.7320+ 0.5740j,
            (1,18) : 0.1640+ 0.1565j,
            (18,19): 1.5042+ 1.3554j,
            (19,20): 0.4095+ 0.4784j,
            (20,21): 0.7089+ 0.9373j,
            (2,22) : 0.4512+ 0.3083j,
            (22,23): 0.8980+ 0.7091j,
            (23,24): 0.8960+ 0.7011j,
            (5,25) : 0.2030+ 0.1034j,
            (25,26): 0.2842+ 0.1447j,
            (26,27): 1.0590+ 0.9337j,
            (27,28): 0.8042+ 0.7006j,
            (28,29): 0.5075+ 0.2585j,
            (29,30): 0.9744+ 0.9630j,
            (30,31): 0.3105+ 0.3619j,
            (31,32): 0.3410+ 0.5302j}.items()}

# Convert impedances to per unit
Z = {key: val / Z_base for key, val in Z_Nominal.items()}

# bus variables
bus_vars = list(range(0,33))

# net injection to power flow mapping
bus2flow = {0  : [(0,1)],
            1  : [(0,1),(1,2),(1,18)],
            18 : [(1,18),(18,19)],
            19 : [(18,19),(19,20)],
            20 : [(19,20),(20,21)],
            21 : [(20,21)],
            2  : [(1,2),(2,3),(2,22)],
            22 : [(2,22),(22,23)],
            23 : [(22,23),(23,24)],
            24 : [(23,24)],
            3  : [(2,3),(3,4)],
            4  : [(3,4),(4,5)],
            5  : [(4,5),(5,6),(5,25)],
            25 : [(5,25),(25,26)],
            26 : [(25,26),(26,27)],
            27 : [(26,27),(27,28)],
            28 : [(27,28),(28,29)],
            29 : [(28,29),(29,30)],
            30 : [(29,30),(30,31)],
            31 : [(30,31),(31,32)],
            32 : [(31,32)],
            6  : [(5,6),(6,7)],
            7  : [(6,7),(7,8)],
            8  : [(7,8),(8,9)],
            9  : [(8,9),(9,10)],
            10 : [(9,10),(10,11)],
            11 : [(10,11),(11,12)],
            12 : [(11,12),(12,13)],
            13 : [(12,13),(13,14)],
            14 : [(13,14),(14,15)],
            15 : [(14,15),(15,16)],
            16 : [(15,16),(16,17)],
            17 : [(16,17)]}

# P/Q loads
load_p_nominal = { key: val for key, val in {
            0 : 0,
            1 : 100e3, 
            2 : 90e3,
            3 : 120e3, 
            4 : 60e3,
            5 : 60e3,
            6 : 200e3,
            7 : 200e3,
            8 : 60e3, 
            9 : 60e3,
            10: 45e3, 
            11: 60e3, 
            12: 60e3, 
            13: 120e3, 
            14: 60e3, 
            15: 60e3, 
            16: 60e3, 
            17: 90e3,
            18: 90e3, 
            19: 90e3,
            20: 90e3,
            21: 90e3, 
            22: 90e3, 
            23: 420e3,
            24: 420e3, 
            25: 60e3, 
            26: 60e3, 
            27: 60e3, 
            28: 120e3, 
            29: 200e3,
            30: 150e3, 
            31: 210e3,
            32: 60e3}.items()}

load_q_nominal = { key: val for key, val in {
            0 : 0,
            1 : 60e3,
            2 : 40e3,
            3 : 80e3,
            4 : 30e3,
            5 : 20e3,
            6 : 100e3,
            7 : 100e3,
            8 : 20e3,
            9 : 20e3,
            10: 30e3,
            11: 35e3,
            12: 35e3,
            13: 80e3,
            14: 10e3,
            15: 20e3,
            16: 20e3,
            17: 40e3,
            18: 40e3,
            19: 40e3,
            20: 40e3,
            21: 40e3,
            22: 50e3,
            23: 200e3,
            24: 200e3,
            25: 25e3,
            26: 25e3,
            27: 20e3,
            28: 70e3,
            29: 600e3,
            30: 70e3,
            31: 100e3,
            32: 40e3}.items()}

# Convert loads to per unit, one value per time step (nominal load at every hour)
load_p = {bus: [P / S_base for _ in range(N)] for bus, P in load_p_nominal.items()}
load_q = {bus: [Q / S_base for _ in range(N)] for bus, Q in load_q_nominal.items()}

## OPF Model

The whole optimization problem is built inside the class `BranchFlow_33bus`. When you create the class, each `define_...` function adds one part of the problem to a Pyomo model called `self.m`.

### Variables (what the solver is solving for)

All of these are defined for every bus or line and for every hour.

| Variable | Meaning |
|---|---|
| `v[bus, t]` | Voltage magnitude at each bus (p.u.) |
| `pline[line, t]` | Active power entering each line at its sending end (W) |
| `qline[line, t]` | Reactive power entering each line at its sending end (var) |
| `Iline[line, t]` | Current magnitude in each line (A) |
| `Pg[0, t]`, `Qg[0, t]` | Active and reactive power supplied by the substation |

The flows and currents are stored in real units (W, var, A) and divided by `S_base` or `I_base` inside the equations to get per unit.

### Constraints (the rules the solution has to follow)

**Root bus** (`define_rootbus_constraints`). The substation voltage is fixed: $v_0 = V_{mag}$.

**Voltage drop along each line** (`define_voltagebalance_constraints`). For a line from bus $i$ to bus $j$ with resistance $r$ and reactance $x$:

$$v_j^2 = v_i^2 - 2(rP + xQ) + (r^2 + x^2) I^2$$

This says the voltage falls as you move away from the substation. The more power a line carries and the higher its impedance, the bigger the drop.

**Power and current in each line** (`define_branchflow_constraint`):

$$P^2 + Q^2 = v_i^2 \, I^2$$

This links the current in a line to the power it carries (apparent power equals voltage times current). It is a quadratic equality, which is what makes the problem nonlinear and is the reason a nonlinear solver like IPOPT is needed.

**Power balance at each bus** (`define_powerbalance_constraints`). At every bus, for both active and reactive power:

(power coming in, minus the losses on the incoming line) + generation - load - power going out = 0

The loss on a line is $r I^2$ for active power and $x I^2$ for reactive power. Only bus 0 has generation (the substation). Every other bus only has load.

**Voltage limits** (`define_voltagelimits_constraints`). Every bus must stay between `V_min` and `V_max`.

### Objective (what is being minimized)

`define_objective_function` adds up the losses in every line over every hour:

$$\min \sum_{t}\sum_{lines} r \, I^2$$

### Why there are no voltage angles

A full AC power flow also tracks the voltage angle at every bus. The branch flow model removes the angles from the equations and works only with voltage magnitudes, line flows and currents. For a radial network like this one, that loses no accuracy, and the equations end up simpler.

### Solver

The last line of `__init__` sets the solver to IPOPT. `solve()` hands the model to IPOPT and returns its report.

In [ ]:
class BranchFlow_33bus():

    def __init__(self):
        self.m = ConcreteModel()
        self.define_variables()
        self.define_rootbus_constraints()
        self.define_voltagebalance_constraints()
        self.define_powerbalance_constraints()
        self.define_branchflow_constraint()
        self.define_objective_function()
        self.define_voltagelimits_constraints()
        self.solver = SolverFactory('ipopt')

    def define_variables(self):
        self.m.time = Set(initialize=t)
        self.m.generators = Set(initialize=[0])
        self.m.v = Var(bus_vars, self.m.time, domain=NonNegativeReals, initialize=Vmag)
        self.m.pline = Var(flow_vars, self.m.time, domain=Reals, initialize=0)
        self.m.qline = Var(flow_vars, self.m.time, domain=Reals, initialize=0)
        self.m.Iline = Var(flow_vars, self.m.time, domain=NonNegativeReals, initialize=0)
        self.m.Pg = Var(self.m.generators, self.m.time, domain=Reals, initialize=0)
        self.m.Qg = Var(self.m.generators, self.m.time, domain=Reals, initialize=0)

    def define_rootbus_constraints(self):
        self.m.rootbus_cons = ConstraintList()
        for hour in self.m.time:
            self.m.rootbus_cons.add(expr=self.m.v[0, hour] == Vmag)

    def define_objective_function(self):
        self.m.obj = Objective(
            expr=quicksum((self.m.Iline[fb, tb, hour]/I_base)**2 * np.real(Z[fb, tb]) for (fb, tb) in flow_vars for hour in self.m.time),
            sense=minimize)

    def define_branchflow_constraint(self):

        self.m.branchflow_cons = ConstraintList()
        for hour in self.m.time:
            for (fb, tb) in flow_vars:
                self.m.branchflow_cons.add(
                    expr=(self.m.pline[fb, tb, hour]/S_base)**2 + (self.m.qline[fb, tb, hour]/S_base)**2 ==
                         (self.m.v[fb, hour])**2 * (self.m.Iline[fb, tb, hour]/I_base)**2)

    def define_voltagebalance_constraints(self):
        self.m.voltagebalance_cons = ConstraintList()
        for hour in self.m.time:
            for (fb, tb) in flow_vars:
                self.m.voltagebalance_cons.add(expr=(self.m.v[tb, hour])**2 == (self.m.v[fb, hour])**2 -
                                               (2 * (self.m.pline[fb, tb, hour]/S_base * np.real(Z[fb, tb]) + self.m.qline[fb, tb, hour]/S_base * np.imag(Z[fb, tb])))
                                               + ((self.m.Iline[fb, tb, hour]/I_base)**2)*((np.real(Z[fb, tb]))**2 + (np.imag(Z[fb, tb]))**2))

    def define_voltagelimits_constraints(self):
        self.m.voltagelimits_cons = ConstraintList()
        for hour in self.m.time:
            for bus in bus_vars:
                self.m.voltagelimits_cons.add(expr=self.m.v[bus, hour] <= V_max)
                self.m.voltagelimits_cons.add(expr=self.m.v[bus, hour] >= V_min)

    def define_powerbalance_constraints(self):
        self.m.powerbalance_cons = ConstraintList()
        for hour in self.m.time:
            for bus in bus2flow:
                if bus == 0:
                    Pg = self.m.Pg[0, hour]/S_base
                    Qg = self.m.Qg[0, hour]/S_base
                else:
                    Pg = 0
                    Qg = 0
                p_out_flow = sum(self.m.pline[fb, tb, hour]/S_base for (fb, tb) in bus2flow[bus] if bus == fb)
                p_in_flow = sum(self.m.pline[fb, tb, hour]/S_base - (np.real(Z[fb, tb]) * (self.m.Iline[fb, tb, hour]/I_base)**2) for (fb, tb) in bus2flow[bus] if bus == tb)
                q_out_flow = sum(self.m.qline[fb, tb, hour]/S_base for (fb, tb) in bus2flow[bus] if bus == fb)
                q_in_flow = sum(self.m.qline[fb, tb, hour]/S_base - (np.imag(Z[fb, tb]) * (self.m.Iline[fb, tb, hour]/I_base)**2) for (fb, tb) in bus2flow[bus] if bus == tb)

                self.m.powerbalance_cons.add(expr=p_in_flow + (Pg - load_p[bus][hour]) - p_out_flow == 0)
                self.m.powerbalance_cons.add(expr=q_in_flow + (Qg - load_q[bus][hour]) - q_out_flow == 0)

    def solve(self):
        self.info = self.solver.solve(self.m)
        return self.info

## Solve

This cell builds the model and solves it. The output is IPOPT's report. The two things to check are:

- `'Termination condition': 'optimal'`, which means IPOPT found a solution that satisfies every constraint.
- `'Number of variables'` and `'Number of constraints'`, which show how big the problem is. With 24 hours it has about 3,100 variables and 4,700 constraints, and it solves in under a second.

`'number of solutions': 0` in the last part of the report is a quirk of how Pyomo prints IPOPT results. It does not mean the solve failed.

In [ ]:
# solve the model
pfm = BranchFlow_33bus()
pfm.solve()

## Store Results

After solving, the values are read back out of the Pyomo model. Writing `pfm.m.v[bus, hour]()` with brackets at the end returns the solved value of that variable.

For each hour, this cell saves:

- `V_buses[hour]`: the voltage at all 33 buses (p.u.)
- `P_flows_dict`, `Q_flows_dict`, `I_flows_dict`: the active power, reactive power and current in all 32 lines
- `Pg_values`: the power supplied by the substation, converted to kW
- `losses_values`: the total line losses (sum of $rI^2$), converted to kW
- `branches_dict`: the same line results grouped by line name, for example `branches_dict[0]["0-1"]` gives P, Q and I for line 0-1 at hour 0

In [ ]:
# Dictionaries to store results
V_buses = {}
P_flows_dict = {}
Q_flows_dict = {}
I_flows_dict = {}
branches_dict = {}
branches = [f"{fb}-{tb}" for (fb, tb) in flow_vars]
Pg_values=[]
losses_values=[]

# Extract and store results for each hour
for hour in t:
    # Bus voltages
    V_buses[hour] = [pfm.m.v[bus, hour]() for bus in bus_vars]
    # Branch flows
    P_flows_dict[hour] = [pfm.m.pline[fb, tb, hour]() for (fb, tb) in flow_vars]
    Q_flows_dict[hour] = [pfm.m.qline[fb, tb, hour]() for (fb, tb) in flow_vars]
    I_flows_dict[hour] = [pfm.m.Iline[fb, tb, hour]() for (fb, tb) in flow_vars]
    Pg_values.append(pfm.m.Pg[0, hour]()/1000)# To make it n KW
    losses_values.append(float((S_base/1000)*sum((pfm.m.Iline[fb, tb, hour]()/I_base)**2 * np.real(Z[fb, tb]) for (fb, tb) in flow_vars))) # in kW
# Store in a combined dictionary
    branches_dict[hour] = {
        branch: {
            "P": P_flows_dict[hour][i],
            "Q": Q_flows_dict[hour][i],
            "I": I_flows_dict[hour][i]
        } for i, branch in enumerate(branches)
    }
    

## Save Results

This cell writes the results to files so you can use them later without solving again. They go into `Results/Obj(MinLosses)/Results_<Vmag>pu/`. The folder name includes the objective and the substation voltage, so runs with different settings don't overwrite each other.

| File | Contents |
|---|---|
| `Voltage_Profiles_Data.csv` | Bus voltages (p.u.). Rows are buses, columns are hours |
| `Pg_Data.pkl` | Substation active power (kW) for each hour |
| `Total_losses_Data.pkl` | Total losses (kW) for each hour |
| `Branches_Data.pkl` | P, Q and I for every line and every hour |

To load a `.pkl` file back into Python:

```python
with open('Results/Obj(MinLosses)/Results_1.0pu/Total_losses_Data.pkl', 'rb') as f:
    losses = pickle.load(f)
```

In [ ]:
# === Create output folder ===
output_folder = f'Results/Obj(MinLosses)/Results_{Vmag}pu'
os.makedirs( output_folder, exist_ok=True)

# Convert voltage results to DataFrame
df_voltages = pd.DataFrame(V_buses, index=bus_vars)
df_voltages.index.name = 'Bus'
# sort columns (hours) numerically
df_voltages = df_voltages[sorted(df_voltages.columns)]
# Build file name
filename = os.path.join(
    output_folder,
    f"Voltage_Profiles_Data.csv"
)
# Save to CSV
df_voltages.to_csv(filename)

# save pg
with open(os.path.join(output_folder, 'Pg_Data.pkl'), "wb") as f:
    pickle.dump(Pg_values, f)
# save losses
with open(os.path.join(output_folder, 'Total_losses_Data.pkl'), "wb") as f:
    pickle.dump(losses_values, f)
# save branch flows
with open(os.path.join(output_folder, 'Branches_Data.pkl'), "wb") as f:
    pickle.dump(branches_dict, f)
print(f"Results saved to '{output_folder}'")

## Quick Check

A short summary of the first hour, so you can compare your results with the published IEEE 33-bus base case. You should get about **202.7 kW** of losses and a minimum voltage of about **0.913 p.u. at bus 17** (bus 18 in 1-indexed numbering).

In [ ]:
hour = 0
V = V_buses[hour]
print(f"Substation supply : {Pg_values[hour]:8.2f} kW")
print(f"Total load        : {sum(load_p_nominal.values())/1e3:8.2f} kW")
print(f"Total losses      : {losses_values[hour]:8.2f} kW")
print(f"Minimum voltage   : {min(V):.4f} p.u. at bus {int(np.argmin(V))}")
print(f"Maximum voltage   : {max(V):.4f} p.u. at bus {int(np.argmax(V))}")

## Voltage Profiles

This plot shows how the voltage at every bus changes over the day.

- The hourly results are interpolated to a 1-minute resolution so the curves look smooth.
- Every bus is drawn as a light gray line.
- The **red** line follows whichever bus has the highest voltage at each moment, and the **blue** line follows whichever bus has the lowest.
- The figure is also saved as `Voltage_Profiles.pdf` in the results folder.

Since the load is the same every hour in this base case, every line is flat. The red line sits at 1.0 p.u. (the substation) and the blue line sits at about 0.913 p.u. (bus 17, the far end of the main feeder). Once you add a load profile or PV, the lines will start to move through the day.

In [ ]:
######## PLOT VOLTAGE PROFILEs ########
# Set font to Times-like
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']


V_list = [V_buses[h] for h in range(N)]

timesteps = 1440  # 1-min resolution
if N == 1:
    time_range = np.linspace(0, 1, timesteps)
else:
    time_range = np.linspace(0, N-1, timesteps)

# Interpolate each bus (cubic needs at least 4 points)
interp_kind = 'cubic' if N >= 4 else 'linear'
V_buses_interp = [[0]*33 for _ in range(timesteps)]
for bus in range(33):
    hourly_voltage = [V_list[h][bus] for h in range(N)]
    if N == 1:
        interpolated = np.full(timesteps, hourly_voltage[0])
    else:
        f = interp1d(np.arange(N), hourly_voltage, kind=interp_kind)
        interpolated = f(time_range)
    for ts in range(timesteps):
        V_buses_interp[ts][bus] = interpolated[ts]

# Identify max & min bus per timestep
max_buses = [np.argmax(V_buses_interp[ts]) for ts in range(timesteps)]
min_buses = [np.argmin(V_buses_interp[ts]) for ts in range(timesteps)]

# --- Main Plot ---
fig, ax = plt.subplots(figsize=(12, 6))
plt.tick_params(axis='both', labelsize=18)

# Plot all buses in gray
for bus in range(33):
    ax.plot(time_range, [V_buses_interp[ts][bus] for ts in range(timesteps)], 
                color='lightgray', linewidth=0.8, alpha=0.5)

# Plot segmented max (red)
prev_bus = max_buses[0]
x_vals = [time_range[0]]
y_vals = [V_buses_interp[0][prev_bus]]
for ts in range(1, timesteps):
    current_bus = max_buses[ts]
    if current_bus == prev_bus:
        x_vals.append(time_range[ts])
        y_vals.append(V_buses_interp[ts][current_bus])
    else:
        ax.plot(x_vals, y_vals, color='red')
        prev_bus = current_bus
        x_vals = [time_range[ts]]
        y_vals = [V_buses_interp[ts][current_bus]]
if x_vals:
    ax.plot(x_vals, y_vals, color='red')

# Plot segmented min (blue)
prev_bus = min_buses[0]
x_vals = [time_range[0]]
y_vals = [V_buses_interp[0][prev_bus]]
for ts in range(1, timesteps):
    current_bus = min_buses[ts]
    if current_bus == prev_bus:
        x_vals.append(time_range[ts])
        y_vals.append(V_buses_interp[ts][current_bus])
    else:
        ax.plot(x_vals, y_vals, color='blue')
        prev_bus = current_bus
        x_vals = [time_range[ts]]
        y_vals = [V_buses_interp[ts][current_bus]]
if x_vals:
    ax.plot(x_vals, y_vals, color='blue')

# Format main plot
    
ax.set_xlabel('Hour of the Day', fontsize=18)
ax.set_ylabel('Voltage (p.u.)', fontsize=18)
ax.set_xticks(np.arange(0, max(N, 1) + 1, 2 if N >= 12 else 1))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
# --- Save or Show ---
plt.tight_layout()
plt.savefig(os.path.join(output_folder, 'Voltage_Profiles.pdf'), dpi=300, bbox_inches='tight')
plt.show()

## Things to Try

- **Use a load profile.** Multiply `load_p` and `load_q` by a daily profile so each hour has a different load. Keep the line impedances the same, since those are the benchmark.
- **Change the horizon.** Set `days = 2` or more in the Configuration cell.
- **Change the substation voltage.** Try `Vmag = 1.02` and see how the losses and the minimum voltage change.
- **Change the objective.** For example, minimize how far the voltages are from 1.0 p.u. instead of minimizing losses.
- **Add something the optimizer can control**, like PV generation at the end-of-feeder buses or a battery with state-of-charge limits. Each one adds a term to the power balance at its bus.

One thing to keep in mind: with fixed loads and a fixed substation voltage, a radial feeder only has one possible power flow solution, so the objective doesn't change the result yet. It starts to matter once you add something the optimizer can choose.